<a href="https://colab.research.google.com/github/labonig/CMSC-487-HW-4/blob/main/_downloads/4e865243430a47a00d551ca0579a6f6c/cifar10_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The code here is borrowed and modified from the [cifar10_tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html).

### 1. Load and normalize CIFAR10


In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:06<00:00, 27.7MB/s]


2. Define a Convolutional Neural Network
========================================


In [3]:
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# Question 0: Basic CNN
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

In [5]:
# Question 1: Deeper CNN architecture
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5, padding='same')
        self.conv2 = nn.Conv2d(6, 16, 7, padding='same')
        self.conv3 = nn.Conv2d(16, 32, 9, padding='same')
        self.conv4 = nn.Conv2d(32, 64, 11, padding='same')
        self.conv5 = nn.Conv2d(64, 128, 13, padding='same')
        self.conv6 = nn.Conv2d(128, 150, 15, padding='same')
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(150 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # shape: 3 x 32 x 32
        # convolutional block 1
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = self.pool(x)
        # shape: 16 x 16 x 16

        # convolutional block 2
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        x = self.pool(x)
        # shape: 64 x 8 x 8

        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))

        x = self.pool(x)
        # shape: 150 x 4 x 4

        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

deeper_net = Net()

3. Define a Loss function and optimizer
=======================================


In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
deep_optimizer = optim.SGD(deeper_net.parameters(), lr=0.001, momentum=0.9)

4. Train the network
====================

In [7]:
def train_network(nn_name, optimizer_name, num_epochs):
  for epoch in range(num_epochs):  # loop over the dataset multiple times

      running_loss = 0.0
      for i, data in enumerate(trainloader, 0):
          # get the inputs; data is a list of [inputs, labels]
          inputs, labels = data

          # zero the parameter gradients
          optimizer_name.zero_grad()

          # forward + backward + optimize
          outputs = nn_name(inputs)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer_name.step()

          # print statistics
          running_loss += loss.item()
          if i % 2000 == 1999:    # print every 2000 mini-batches
              print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
              running_loss = 0.0

  print('Finished Training')


In [8]:
# Training the tutorial network
train_network(net, optimizer, 10)

[1,  2000] loss: 2.243
[1,  4000] loss: 1.886
[1,  6000] loss: 1.695
[1,  8000] loss: 1.605
[1, 10000] loss: 1.536
[1, 12000] loss: 1.499
[2,  2000] loss: 1.433
[2,  4000] loss: 1.402
[2,  6000] loss: 1.362
[2,  8000] loss: 1.347
[2, 10000] loss: 1.328
[2, 12000] loss: 1.313
[3,  2000] loss: 1.237
[3,  4000] loss: 1.243
[3,  6000] loss: 1.228
[3,  8000] loss: 1.231
[3, 10000] loss: 1.194
[3, 12000] loss: 1.214
[4,  2000] loss: 1.142
[4,  4000] loss: 1.143
[4,  6000] loss: 1.135
[4,  8000] loss: 1.138
[4, 10000] loss: 1.148
[4, 12000] loss: 1.121
[5,  2000] loss: 1.054
[5,  4000] loss: 1.051
[5,  6000] loss: 1.046
[5,  8000] loss: 1.068
[5, 10000] loss: 1.075
[5, 12000] loss: 1.082
[6,  2000] loss: 0.987
[6,  4000] loss: 0.972
[6,  6000] loss: 1.004
[6,  8000] loss: 1.040
[6, 10000] loss: 1.019
[6, 12000] loss: 1.015
[7,  2000] loss: 0.924
[7,  4000] loss: 0.954
[7,  6000] loss: 0.955
[7,  8000] loss: 0.976
[7, 10000] loss: 0.956
[7, 12000] loss: 0.977
[8,  2000] loss: 0.849
[8,  4000] 

In [9]:
# save the trained model:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

In [10]:
# Training the deeper model with convolutional blocks
train_network(deeper_net, deep_optimizer, 10)


[1,  2000] loss: 2.211
[1,  4000] loss: 1.902
[1,  6000] loss: 1.716
[1,  8000] loss: 1.600
[1, 10000] loss: 1.543
[1, 12000] loss: 1.483
[2,  2000] loss: 1.425
[2,  4000] loss: 1.396
[2,  6000] loss: 1.357
[2,  8000] loss: 1.340
[2, 10000] loss: 1.311
[2, 12000] loss: 1.301
[3,  2000] loss: 1.227
[3,  4000] loss: 1.233
[3,  6000] loss: 1.225
[3,  8000] loss: 1.190
[3, 10000] loss: 1.213
[3, 12000] loss: 1.220
[4,  2000] loss: 1.134
[4,  4000] loss: 1.137
[4,  6000] loss: 1.141
[4,  8000] loss: 1.125
[4, 10000] loss: 1.118
[4, 12000] loss: 1.114
[5,  2000] loss: 1.050
[5,  4000] loss: 1.061
[5,  6000] loss: 1.058
[5,  8000] loss: 1.073
[5, 10000] loss: 1.067
[5, 12000] loss: 1.065
[6,  2000] loss: 0.983
[6,  4000] loss: 1.020
[6,  6000] loss: 0.995
[6,  8000] loss: 1.024
[6, 10000] loss: 1.034
[6, 12000] loss: 1.009
[7,  2000] loss: 0.936
[7,  4000] loss: 0.959
[7,  6000] loss: 0.984
[7,  8000] loss: 0.967
[7, 10000] loss: 0.962
[7, 12000] loss: 0.979
[8,  2000] loss: 0.911
[8,  4000] 

In [11]:
# save the trained model:
PATH2 = './cifar_deeper_net.pth'
torch.save(deeper_net.state_dict(), PATH2)

In [17]:
# calculates the percentage of correct predictions on either the training or testing dataset
def test_network(nn_name, dataset_name):
  correct = 0
  total = 0
  # since we're not training, we don't need to calculate the gradients for our outputs
  with torch.no_grad():
      for data in dataset_name:
          images, labels = data
          # calculate outputs by running images through the network
          outputs = nn_name(images)
          # the class with the highest energy is what we choose as prediction
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  return (100 * correct // total)

In [13]:
# training accuracy of the tutorial cnn
training_score = test_network(net, trainloader)
print(f'Training accuracy: {training_score} %')

Training accuracy: 71 %


In [14]:
# training accuracy of the deeper cnn
training_score = test_network(deeper_net, trainloader)
print(f'Training accuracy: {training_score} %')

Training accuracy: 71 %


5. Test the network on the test data
====================================

In [15]:
dataiter = iter(testloader)
images, labels = next(dataiter)

In [16]:
# loading back the saved model:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [18]:
# Testing the tutorial network
test_score = test_network(net, testloader)
print(f'Accuracy of the network on the 10000 test images: {test_score} %')

Accuracy of the network on the 10000 test images: 61 %


In [19]:
# Testing the deeper network
test_score = test_network(deeper_net, testloader)
print(f'Accuracy of the network on the 10000 test images: {test_score} %')

Accuracy of the network on the 10000 test images: 60 %


In [20]:
def print_accuracy(nn_name):
  # prepare to count predictions for each class
  correct_pred = {classname: 0 for classname in classes}
  total_pred = {classname: 0 for classname in classes}

  # again no gradients needed
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          outputs = nn_name(images)
          _, predictions = torch.max(outputs, 1)
          # collect the correct predictions for each class
          for label, prediction in zip(labels, predictions):
              if label == prediction:
                  correct_pred[classes[label]] += 1
              total_pred[classes[label]] += 1

  # print accuracy for each class
  for classname, correct_count in correct_pred.items():
      accuracy = 100 * float(correct_count) / total_pred[classname]
      print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

In [21]:
# print the per-class accuracy of the tutorial network
print_accuracy(net)

Accuracy for class: plane is 60.0 %
Accuracy for class: car   is 75.3 %
Accuracy for class: bird  is 51.2 %
Accuracy for class: cat   is 45.2 %
Accuracy for class: deer  is 58.7 %
Accuracy for class: dog   is 50.6 %
Accuracy for class: frog  is 79.4 %
Accuracy for class: horse is 57.6 %
Accuracy for class: ship  is 71.0 %
Accuracy for class: truck is 67.9 %


In [22]:
# print the per-class accuracy of the deeper network
print_accuracy(deeper_net)

Accuracy for class: plane is 68.2 %
Accuracy for class: car   is 69.5 %
Accuracy for class: bird  is 51.0 %
Accuracy for class: cat   is 48.5 %
Accuracy for class: deer  is 54.9 %
Accuracy for class: dog   is 43.5 %
Accuracy for class: frog  is 64.3 %
Accuracy for class: horse is 70.7 %
Accuracy for class: ship  is 77.5 %
Accuracy for class: truck is 61.1 %


In [23]:
del dataiter